# T1: Learned Watermark Diagnostics
**Purpose:** Load trained learned-model checkpoints from Drive, run no-attack evaluation on
100 test images, and run a 100-image overfit test to diagnose pipeline correctness.

**Runtime:** GPU (T4). Estimated time: ~25 minutes.

**Inputs (required on Drive):**
- `MyDrive/watermark_project/project_bundle_v2_<date>.zip` — project bundle
- `MyDrive/watermark_project/checkpoints/main_stage1/latest_checkpoint.pt`
- `MyDrive/watermark_project/checkpoints/learned_control_low/latest_checkpoint.pt`
- `MyDrive/watermark_project/checkpoints/learned_control_mid/latest_checkpoint.pt`
- `MyDrive/watermark_project/checkpoints/learned_control_high/latest_checkpoint.pt`

**Outputs (saved to Drive):**
- `no_attack_eval.csv` — columns: method, image_id, bit_acc
- `overfit_history.json` — per-epoch loss and bit_acc for the overfit sanity test

**Return:** `results_T1_v1_<timestamp>.zip` → place in `incoming/T1/`


In [ ]:
import subprocess, sys, os, time, json, zipfile, hashlib
from pathlib import Path
from datetime import datetime

# ── Flags ────────────────────────────────────────────────────────────────────
SMOKE = False          # Set True to run 3-image smoke test only
N_EVAL_IMAGES = 3 if SMOKE else 100
N_OVERFIT_IMAGES = 3 if SMOKE else 20
OVERFIT_EPOCHS = 10 if SMOKE else 50
N_BITS = 32
SECRET_KEY = 42
PSNR_TARGET = 40.0
# ─────────────────────────────────────────────────────────────────────────────

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/watermark_project')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# GPU check
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip()
print('GPU:', gpu_info)

# Install deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'diffusers', 'transformers', 'accelerate',
    'compressai', 'lpips', 'scipy', 'scikit-image', 'pandas', 'tqdm'
], check=True)

import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# ── Unzip Bundle ─────────────────────────────────────────────────────────────
REPO = Path('/content/repo')
REPO.mkdir(exist_ok=True)

bundle_zips = sorted(DRIVE_DIR.glob('project_bundle_v2_*.zip'))
if not bundle_zips:
    raise FileNotFoundError(
        f'No project_bundle_v2_*.zip found in {DRIVE_DIR}. '
        'Please upload the bundle to Drive first.'
    )
bundle_zip = bundle_zips[-1]  # most recent
print(f'Using bundle: {bundle_zip.name}')

with zipfile.ZipFile(bundle_zip, 'r') as zf:
    zf.extractall(REPO)
print(f'Bundle extracted to {REPO}')

# Add repo to sys.path so we can import track_* and core
sys.path.insert(0, str(REPO))
print('sys.path[0]:', sys.path[0])


In [ ]:
# ── Load Checkpoints ─────────────────────────────────────────────────────────
from track_c.model import WatermarkEncoder, WatermarkDecoder, BandAllocationLayer

CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
CHECKPOINTS = {
    'learned_main_surrogate':  CHECKPOINT_DIR / 'main_stage1'      / 'latest_checkpoint.pt',
    'learned_control_low':     CHECKPOINT_DIR / 'learned_control_low'  / 'latest_checkpoint.pt',
    'learned_control_mid':     CHECKPOINT_DIR / 'learned_control_mid'  / 'latest_checkpoint.pt',
    'learned_control_high':    CHECKPOINT_DIR / 'learned_control_high' / 'latest_checkpoint.pt',
}

REGIME_MAP = {
    'learned_main_surrogate': None,
    'learned_control_low':    'low',
    'learned_control_mid':    'mid',
    'learned_control_high':   'high',
}

loaded_models = {}  # method -> (encoder, decoder)

for method, ckpt_path in CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f'  ⚠ Checkpoint not found, skipping: {ckpt_path}')
        continue
    regime = REGIME_MAP[method]
    enc = WatermarkEncoder(n_bits=N_BITS, fixed_regime=regime).to(device)
    dec = WatermarkDecoder(n_bits=N_BITS).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    enc.load_state_dict(ckpt['encoder_state_dict'])
    dec.load_state_dict(ckpt['decoder_state_dict'])
    enc.eval(); dec.eval()
    loaded_models[method] = (enc, dec)
    print(f'  ✓ Loaded {method}')

print(f'Loaded {len(loaded_models)} model(s): {list(loaded_models.keys())}')


In [ ]:
# ── Load Test Images ─────────────────────────────────────────────────────────
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
import glob

TEST_DIR = REPO / 'data' / 'processed' / 'test'
if not TEST_DIR.exists():
    # Fallback: look for data dir
    TEST_DIR = Path('/content/data/processed/test')
    TEST_DIR.mkdir(parents=True, exist_ok=True)

transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
])

def load_images_from_dir(d: Path, n: int):
    imgs = sorted(list(d.glob('*.png')) + list(d.glob('*.jpg')))[:n]
    tensors, ids = [], []
    for p in imgs:
        img = Image.open(p).convert('RGB')
        tensors.append(transform(img))
        ids.append(p.stem)
    if not tensors:
        raise FileNotFoundError(f'No images found in {d}')
    return torch.stack(tensors), ids  # (N, 3, 256, 256)

print(f'Loading up to {N_EVAL_IMAGES} test images from {TEST_DIR}')
test_imgs, test_ids = load_images_from_dir(TEST_DIR, N_EVAL_IMAGES)
print(f'Loaded {len(test_ids)} images. Shape: {test_imgs.shape}')


In [ ]:
# ── No-Attack Bit Accuracy Evaluation ────────────────────────────────────────
import pandas as pd
from track_b.watermark import ClassicalSpreadSpectrumWatermarker

no_attack_rows = []

# Classical Band 2 sanity reference
print('\n── Classical Band 2 (sanity reference) ──')
wm_classical = ClassicalSpreadSpectrumWatermarker(
    n_bits=N_BITS, secret_key=SECRET_KEY, device=device
)
payload = torch.randint(0, 2, (N_BITS,), dtype=torch.float32)

for i, (img, img_id) in enumerate(zip(test_imgs, test_ids)):
    x = img.unsqueeze(0).to(device)  # (1, 3, 256, 256)
    with torch.no_grad():
        wm_img = wm_classical.embed(x, payload.unsqueeze(0), band_idx=2, psnr_db=PSNR_TARGET, round_8bit=True)
        hard_bits, _ = wm_classical.decode(wm_img, band_idx=2)
    bit_acc = (hard_bits.squeeze(0).float() == payload.to(device)).float().mean().item()
    no_attack_rows.append({'method': 'classical_band2', 'image_id': img_id, 'bit_acc': bit_acc})

mean_ba = sum(r['bit_acc'] for r in no_attack_rows) / max(len(no_attack_rows), 1)
print(f'  Classical Band 2 no-attack mean bit_acc: {mean_ba:.4f} (expected ~0.98+)')

# Learned models
for method, (enc, dec) in loaded_models.items():
    print(f'\n── {method} ──')
    for img, img_id in zip(test_imgs, test_ids):
        x = img.unsqueeze(0).to(device)
        payload_b = torch.randint(0, 2, (1, N_BITS), dtype=torch.float32).to(device)
        with torch.no_grad():
            wm_img, _ = enc(x, payload_b)
            logits = dec(wm_img)
            decoded_bits = (logits > 0.0).float()
        bit_acc = (decoded_bits == payload_b).float().mean().item()
        no_attack_rows.append({'method': method, 'image_id': img_id, 'bit_acc': bit_acc})

no_attack_df = pd.DataFrame(no_attack_rows)
print('\nNo-attack evaluation summary:')
print(no_attack_df.groupby('method')['bit_acc'].agg(['mean', 'min', 'max']))

# Save to Drive
no_attack_csv = DRIVE_DIR / 'no_attack_eval.csv'
no_attack_df.to_csv(no_attack_csv, index=False)
print(f'\nSaved: {no_attack_csv}')


In [ ]:
# ── Overfit Test ──────────────────────────────────────────────────────────────
# Train encoder+decoder for OVERFIT_EPOCHS on N_OVERFIT_IMAGES, identity noise, 32-bit payload.
# Bit accuracy should reach >95% by epoch 20 if the pipeline is correct.
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'\nRunning overfit test: {OVERFIT_EPOCHS} epochs on {N_OVERFIT_IMAGES} images')
overfit_imgs = test_imgs[:N_OVERFIT_IMAGES].to(device)
fixed_payload = torch.randint(0, 2, (N_OVERFIT_IMAGES, N_BITS), dtype=torch.float32).to(device)

# Use fresh models for overfit test (don't modify loaded checkpoints)
enc_ov = WatermarkEncoder(n_bits=N_BITS, fixed_regime=None).to(device)
dec_ov = WatermarkDecoder(n_bits=N_BITS).to(device)
enc_ov.train(); dec_ov.train()

optimizer = optim.Adam(list(enc_ov.parameters()) + list(dec_ov.parameters()), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

history = {'loss': [], 'bit_acc': []}

for epoch in range(OVERFIT_EPOCHS):
    optimizer.zero_grad()
    wm_imgs = enc_ov(overfit_imgs, fixed_payload)
    logits = dec_ov(wm_imgs)
    loss = criterion(logits, fixed_payload)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        decoded = (torch.sigmoid(logits) > 0.5).float()
        bit_acc = (decoded == fixed_payload).float().mean().item()
    history['loss'].append(loss.item())
    history['bit_acc'].append(bit_acc)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:3d}/{OVERFIT_EPOCHS}  loss={loss.item():.4f}  bit_acc={bit_acc:.4f}')

final_bit_acc = history['bit_acc'][-1]
print(f'\nFinal overfit bit_acc: {final_bit_acc:.4f}')
if final_bit_acc > 0.95:
    print('✓ PASS: Pipeline is correct (bit_acc > 0.95)')
else:
    print('✗ WARN: Pipeline may have a bug (bit_acc <= 0.95 after overfit)')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(history['loss']); ax1.set_title('Overfit Loss'); ax1.set_xlabel('Epoch')
ax2.plot(history['bit_acc']); ax2.axhline(0.95, color='r', linestyle='--', label='0.95')
ax2.set_title('Overfit Bit Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plot_path = '/tmp/overfit_plot.png'
plt.savefig(plot_path, dpi=100)
plt.show()
print(f'Plot saved: {plot_path}')

# Save history to Drive
overfit_path = DRIVE_DIR / 'overfit_history.json'
with open(overfit_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Saved: {overfit_path}')


In [ ]:
# ── Create Return Package ────────────────────────────────────────────────────
ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
zip_path = DRIVE_DIR / f'results_T1_v1_{ts}.zip'

files_to_include = [
    (DRIVE_DIR / 'no_attack_eval.csv', 'no_attack_eval.csv'),
    (DRIVE_DIR / 'overfit_history.json', 'overfit_history.json'),
    (Path('/tmp/overfit_plot.png'), 'overfit_plot.png'),
]

checksums = {}
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, arcname in files_to_include:
        if Path(src).exists():
            zf.write(src, arcname)
            checksums[arcname] = hashlib.sha256(Path(src).read_bytes()).hexdigest()
            print(f'  + {arcname}')
        else:
            print(f'  ⚠ Not found, skipping: {src}')

    manifest = {
        'task': 'T1', 'version': 'v1',
        'timestamp': ts,
        'files': [a for _, a in files_to_include],
        'checksums': checksums,
        'summary': {
            'gpu': gpu_info,
            'n_eval_images': N_EVAL_IMAGES,
            'n_overfit_images': N_OVERFIT_IMAGES,
            'overfit_epochs': OVERFIT_EPOCHS,
            'final_overfit_bit_acc': float(history['bit_acc'][-1]) if history['bit_acc'] else None,
        }
    }
    zf.writestr('manifest.json', json.dumps(manifest, indent=2))

print(f'\nPackage saved: {zip_path}')
print('Return this file: place it in incoming/T1/')
